<a href="https://colab.research.google.com/github/Naveen-gale/DATA-SCENS-LEARNING/blob/main/novelai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
!pip install -q transformers datasets accelerate sentencepiece

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
BASE_PATH = "/content/drive/MyDrive/AntigoneGPT"

MODEL_SAVE_PATH = f"{BASE_PATH}/final_model"

DATASET_PATH = f"{BASE_PATH}/final_dataset.jsonl"

In [3]:
!pip install -U -q transformers datasets accelerate peft bitsandbytes trl sentencepiece "torchao>=0.16.0"

### 1. Configuration & Model Loading
We initialize 4-bit quantization using `bitsandbytes` to fit the model on a T4 GPU and load the Qwen2.5-0.5B-Instruct model with PEFT requirements.

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit Quantization Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Comprehensive targeting for Qwen
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


### 2. Dataset Processing with Chat ML Template
We load the JSONL, validate the entries, and format them into the `<|im_start|>` format used by Qwen.

In [5]:
import json
import torch
from datasets import Dataset

# Standard production path definitions
BASE_PATH = "/content/drive/MyDrive/AntigoneGPT"
DATASET_PATH = f"{BASE_PATH}/final_dataset.jsonl"

# Ensure 'data' is loaded correctly
data = []
try:
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
except FileNotFoundError:
    print(f"Error: File not found at {DATASET_PATH}. Please verify your Google Drive mount.")

raw_samples = []
seen = set()
for obj in data:
    instr, resp = obj.get("instruction", "").strip(), obj.get("response", "").strip()
    if instr and resp:
        pair = (instr, resp)
        if pair not in seen:
            raw_samples.append({"instruction": instr, "response": resp})
            seen.add(pair)

def apply_chat_template(sample):
    messages = [
        {"role": "system", "content": "You are an expert English Literature tutor specializing in Sophocles' Antigone. Answer only using knowledge learned during fine-tuning. If the dataset does not contain enough information, say you do not know instead of inventing facts."},
        {"role": "user", "content": sample["instruction"]},
        {"role": "assistant", "content": sample["response"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

if raw_samples:
    dataset = Dataset.from_list(raw_samples).shuffle(seed=42)
    dataset = dataset.map(apply_chat_template)
    split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
    print(f"Successfully prepared {len(train_dataset)} training samples for Qwen.")
else:
    print("Dataset preparation failed. Please check the source file.")

Map:   0%|          | 0/1549 [00:00<?, ? examples/s]

Successfully prepared 1471 training samples for Qwen.


### 3. Training & Model Merging
We use the SFTTrainer for optimized instruction tuning and merge the LoRA weights at the end.

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, AutoModelForCausalLM
from peft import PeftModel
import torch
import os

# Production path definitions
BASE_PATH = "/content/drive/MyDrive/AntigoneGPT"
MODEL_SAVE_PATH = os.path.abspath(f"{BASE_PATH}/final_model")

# Ensure datasets are available in global scope
train_ds = globals().get('train_dataset')
eval_ds = globals().get('eval_dataset')

if train_ds is None:
    raise ValueError("train_dataset not found. Please run cell b5e4821a first.")

# Configuration for training - Qwen Optimized
# Removed max_seq_length to satisfy the SFTConfig init signature
sft_config = SFTConfig(
    output_dir="./qwen-antigone-checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    num_train_epochs=3,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="tensorboard",
    weight_decay=0.01,
    dataset_text_field="text",
    packing=True
)

# SFTTrainer setup
# We rely on packing=True in SFTConfig to handle sequence lengths automatically
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting Qwen QLoRA Training with Packing enabled...")
trainer.train()

# Merge LoRA adapters into base model
print("Merging LoRA weights into base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(base_model, trainer.state.best_model_checkpoint)
merged_model = merged_model.merge_and_unload()

# Final Save
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
merged_model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"Production model successfully saved to {MODEL_SAVE_PATH}")

Adding EOS to train dataset:   0%|          | 0/1471 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1471 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1471 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/1471 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/78 [00:00<?, ? examples/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Merging LoRA weights into base model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Production model successfully saved to /content/drive/MyDrive/AntigoneGPT/final_model


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./qwen-antigone-checkpoints/runs

### 4. Production Inference & Testing
Testing the model with the requested evaluation prompts using specialized generation parameters to minimize hallucinations.

In [7]:
import torch
import os
from transformers import pipeline, AutoTokenizer

# Using absolute paths to avoid Repo ID validation errors
BASE_PATH = "/content/drive/MyDrive/AntigoneGPT"
MODEL_SAVE_PATH = os.path.abspath(f"{BASE_PATH}/final_model")

test_prompts = [
    "Who is Antigone?", "Who is Creon?", "Who is Ismene?",
    "Why did Antigone bury Polyneices?", "Summarize Antigone.",
    "Explain Creon's character.", "Give a 2 mark answer on Antigone.",
    "Give a 5 mark answer on Creon.", "Explain the ending of Antigone.",
    "Explain Antigone in simple English.",
    "give all 2 markes qution in sene 1 withe answre "
]

if os.path.exists(MODEL_SAVE_PATH):
    print(f"Loading fine-tuned model from: {MODEL_SAVE_PATH}...")
    try:
        pipe = pipeline(
            "text-generation",
            model=MODEL_SAVE_PATH,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)

        print("\n--- PRODUCTION TEST RESULTS ---\n")
        for q in test_prompts:
            messages = [{"role": "user", "content": q}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

            out = pipe(
                prompt,
                max_new_tokens=256,
                do_sample=True,
                temperature=0.1,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id
            )

            response = out[0]['generated_text'].split("<|im_start|>assistant")[-1].strip().replace("<|im_end|>", "")
            print(f"Q: {q}\nA: {response}\n{'-'*50}")
    except Exception as e:
        print(f"Test failed: {e}")
else:
    print("Model directory not found. Please ensure the training and merge cell completed successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading fine-tuned model from: /content/drive/MyDrive/AntigoneGPT/final_model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'top_p', 'max_new_tokens', 'pad_token_id', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- PRODUCTION TEST RESULTS ---



...

Q: Give a 5 mark answer on Creon.
A: Creon is a character in Sophocles' Antigone who embodies the tragic and complex nature of human morality and justice. He is a figure who struggles with his own sense of right and wrong, and who ultimately succumbs to the forces of fate and circumstance. Creon's character serves as a cautionary tale about the dangers of hubris and pride, and highlights the importance of humility and respect for authority.
--------------------------------------------------


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Explain the ending of Antigone.
A: The ending of "Antigone" is a tragic and complex one that explores themes of fate, morality, and human nature. The story ends with Creon's death, which leaves the characters in a state of confusion and despair. The final scene sees the characters reflecting on their own actions and decisions, highlighting the consequences of their choices and the impact they have on those around them. The ending also serves to underscore the idea that even in the face of overwhelming odds, there is still hope for redemption and forgiveness.
--------------------------------------------------


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Explain Antigone in simple English.
A: Antigone is a character from Sophocles' "Antigone," who embodies the virtues of courage and loyalty. She is a woman who stands up for what she believes to be right, even when it goes against societal norms or personal feelings. Her character is characterized by her strong sense of justice and her willingness to fight for what is just, even if it means sacrificing herself. Antigone's story serves as a powerful reminder that even in the face of adversity, one can find strength and resolve through their own actions and convictions.
--------------------------------------------------
Q: give all 2 markes qution in sene 1 withe answre 
A: Sure! Here is the question and answer for Question 1:

**Question:** What is the difference between a function and a method in Python?

**Answer:** In Python, functions are defined using the `def` keyword followed by the function name and parentheses containing any arguments or parameters that will be passed to the 

### Reloading and Reprocessing Data

Since you've added new data, we need to reload and reprocess the dataset to incorporate the changes. This ensures our model trains on the most current information.

In [ ]:
import json

# Initialize an empty list to store the data
data = []

# Open the dataset file and load each JSON line
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# Print the total number of samples and the first sample to verify
print(f"Total number of samples: {len(data)}")
print(f"First sample: {data[0]}")

In [ ]:
# Prepare the data in the required "User: Question\n\nAssistant: Answer" format
texts = []

for row in data:
    sample = f"""User: {row['instruction']}

Assistant: {row['response']}"""
    texts.append(sample)

# Print the first formatted text to verify
print("First formatted text sample:")
print(texts[0])

In [ ]:
from datasets import Dataset

# Create a Hugging Face Dataset from the prepared texts
dataset = Dataset.from_dict({"text": texts})

# Display the dataset information
display(dataset)

In [ ]:
import math

def tokenize(example):
    # Tokenize the input text, truncating to max_length and padding to max_length
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

# Apply the tokenization function to the entire dataset
tokenized = dataset.map(tokenize, batched=True)

# Display the tokenized dataset information
display(tokenized)

# Calculate and print the approximate number of training steps for warmup_steps calculation
train_samples = math.ceil(len(tokenized) * (1 - 0.05)) # Assuming 0.05 test_size
approx_total_steps = (5 * train_samples) / 2 # epochs * (train_samples / batch_size)
print(f"Approximate total training steps: {approx_total_steps}")

In [ ]:
# Split the tokenized dataset into training and evaluation sets
split = tokenized.train_test_split(test_size=0.05, seed=42) # Added seed for reproducibility

train_dataset = split["train"]

eval_dataset = split["test"]

# Print information about the training and evaluation datasets
print("Training Dataset:")
print(train_dataset)

print("Evaluation Dataset:")
print(eval_dataset)

In [ ]:
import json

data = []

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

print(len(data))
print(data[0])

In [ ]:
texts = []

for row in data:

    sample = f"""User: {row['instruction']}

Assistant: {row['response']}"""

    texts.append(sample)

print(texts[0])

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": texts})
dataset

In [ ]:
from transformers import GPT2TokenizerFast

In [ ]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

tokenized = dataset.map(tokenize, batched=True)

tokenized

In [ ]:
split = tokenized.train_test_split(test_size=0.05)

train_dataset = split["train"]

eval_dataset = split["test"]

print(train_dataset)

print(eval_dataset)

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")

model.resize_token_embeddings(len(tokenizer))

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(

    output_dir="./checkpoints",

    num_train_epochs=5, # Reduced from 10 to 5 to combat overfitting

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    learning_rate=3e-5, # Slightly reduced learning rate

    weight_decay=0.05, # Increased weight decay for regularization

    warmup_steps=184, # Calculated as 5% of approx_total_steps (3680 * 0.05)

    logging_steps=25,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=True,

    report_to="none"
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling, EarlyStoppingCallback

data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=False
)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=eval_dataset,

    data_collator=data_collator,

    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] # Add EarlyStoppingCallback
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(MODEL_SAVE_PATH)

tokenizer.save_pretrained(MODEL_SAVE_PATH)

print("Model Saved Successfully!")

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MODEL_SAVE_PATH,
    tokenizer=MODEL_SAVE_PATH
)

prompt = """User: Who is Antigone?

Assistant:"""

result = pipe(
    prompt,
    max_new_tokens=150,
    do_sample=False,
    temperature=0.3
)

print(result[0]["generated_text"])

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./qwen-antigone-checkpoints/runs